In [1]:
# %%
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

# Usa PROJECT_ROOT (che è ".." rispetto a scripts/) per puntare alle cartelle giuste
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
OUT_ROOT = os.path.join(PROJECT_ROOT, "scripts", "data_precomputed")

# Verifica subito se vede i file (per debug)
import glob
test_files = glob.glob(os.path.join(DATA_ROOT, "**/*.wav"), recursive=True)
print(f"File trovati in {DATA_ROOT}: {len(test_files)}")


Project root: c:\Users\eleon\Desktop\PROGETTO_ML\SER-Machine-Learning-Project
File trovati in c:\Users\eleon\Desktop\PROGETTO_ML\SER-Machine-Learning-Project\data: 1440


In [2]:
# %%
import os
import sys
import random
import torch
import torchaudio.transforms as T
import torchaudio.functional as F
import soundfile as sf
from tqdm import tqdm

# 1. Definiamo la radice del progetto risalendo di un livello (..) 
# rispetto alla posizione di questo notebook (scripts/)
PROJECT_ROOT = os.path.abspath("..")

# 2. Aggiungiamo PROJECT_ROOT al sys.path se non c'è già
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# 3. Ora gli import da 'src' funzioneranno correttamente
from src.preprocessing.dataset import (
    list_ravdess_files,
    filter_audio_speech,
    extract_label_idx,
    parse_ravdess_filename,
)

print(f"Project root impostata a: {PROJECT_ROOT}")

Project root impostata a: c:\Users\eleon\Desktop\PROGETTO_ML\SER-Machine-Learning-Project


In [3]:
# %%
# Usa percorsi assoluti basati su PROJECT_ROOT
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
OUT_ROOT = os.path.join(PROJECT_ROOT, "scripts", "data_precomputed")

SAMPLE_RATE = 16000
N_MELS = 64
MAX_DURATION = 4.0

TIME_SHIFT_S = 0.03
N_AUG = 3


In [4]:
# %%
os.makedirs(OUT_ROOT, exist_ok=True)
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUT_ROOT, split), exist_ok=True)


In [5]:
# %%
train_spk = [
    '01','02','03','04','05','07','08','09',
    '10','11','13','14','17','19','20','21','22','23'
]
val_spk = ['15','16']
test_spk = ['06','12','18','24']

def split_by_speakers(files):
    out = {"train": [], "val": [], "test": []}
    for fp in files:
        actor = parse_ravdess_filename(fp)["actor"]
        if actor in train_spk:
            out["train"].append(fp)
        elif actor in val_spk:
            out["val"].append(fp)
        elif actor in test_spk:
            out["test"].append(fp)
    return out


In [6]:
# %%
mel = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=1024,
    hop_length=256,
    win_length=1024,
    n_mels=N_MELS,
    power=2.0,
)

to_db = T.AmplitudeToDB(stype="power", top_db=80.0)
max_samples = int(SAMPLE_RATE * MAX_DURATION)

# Configurazione Augmentation identica al tuo modello originale
aug_cfg = {
    "pitch_shift": True,
    "gain": True,
    "gain_db": (-3, 3),
    "time_shift": True,
    "time_shift_s": 0.03,
    "noise": True,
    "snr_db": (25, 40),
}

def load_wav(path):
    wav_np, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = torch.from_numpy(wav_np).transpose(0, 1)
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        wav = F.resample(wav, sr, SAMPLE_RATE)
    if wav.size(1) < max_samples:
        wav = torch.nn.functional.pad(wav, (0, max_samples - wav.size(1)))
    else:
        wav = wav[:, :max_samples]
    wav = wav / (wav.abs().max() + 1e-9)
    return wav

def apply_heavy_augmentation(wav, sr, cfg):
    # 1. Pitch Shift (Lento, fatto una volta sola qui)
    if cfg.get("pitch_shift", False):
        n_steps = random.randint(-2, 2)
        if n_steps != 0:
            wav = T.PitchShift(sr, n_steps)(wav)
    # 2. Random Gain
    if cfg.get("gain", False):
        gmin, gmax = cfg.get("gain_db", (-3, 3))
        gain_db = random.uniform(gmin, gmax)
        wav = wav * (10 ** (gain_db / 20))
    # 3. Time Shift
    if cfg.get("time_shift", False):
        max_shift = int(cfg.get("time_shift_s", 0.03) * sr)
        shift = random.randint(-max_shift, max_shift)
        wav = torch.roll(wav, shifts=shift, dims=1)
    # 4. Noise
    if cfg.get("noise", False):
        snr_db = random.uniform(*cfg.get("snr_db", (25, 40)))
        sig_pow = wav.pow(2).mean()
        noise = torch.randn_like(wav)
        noise_pow = noise.pow(2).mean()
        scale = torch.sqrt(sig_pow / (10**(snr_db/10) * noise_pow + 1e-9))
        wav = wav + scale * noise
    return torch.clamp(wav, -1.0, 1.0)

In [7]:
# %%
files = filter_audio_speech(list_ravdess_files(DATA_ROOT))
splits = split_by_speakers(files)
counter = {"train": 0, "val": 0, "test": 0}

for split, file_list in splits.items():
    print(f"\nProcessing {split} ({len(file_list)} files)")
    for fp in tqdm(file_list):
        label = extract_label_idx(fp)
        wav_orig = load_wav(fp)

        # Salvataggio file originale
        spec = to_db(mel(wav_orig))
        spec = (spec - spec.mean()) / (spec.std() + 1e-6)
        torch.save({"spec": spec, "label": label}, 
                   os.path.join(OUT_ROOT, split, f"{counter[split]}.pt"))
        counter[split] += 1

        # Generazione aumentata (solo per il set di TRAIN)
        if split == "train":
            for _ in range(N_AUG):
                # Augmentation audio
                wav_aug = apply_heavy_augmentation(wav_orig, SAMPLE_RATE, aug_cfg)
                # Spettrogramma
                spec = to_db(mel(wav_aug))
                # SpecAugment (Frequency & Time masking)
                if random.random() < 0.5:
                    spec = T.FrequencyMasking(10)(spec)
                    spec = T.TimeMasking(25)(spec)
                # Normalizzazione e salvataggio
                spec = (spec - spec.mean()) / (spec.std() + 1e-6)
                torch.save({"spec": spec, "label": label}, 
                           os.path.join(OUT_ROOT, split, f"{counter[split]}.pt"))
                counter[split] += 1



Processing train (1080 files)


100%|██████████| 1080/1080 [55:03<00:00,  3.06s/it] 



Processing val (120 files)


100%|██████████| 120/120 [00:00<00:00, 207.23it/s]



Processing test (240 files)


100%|██████████| 240/240 [00:01<00:00, 190.84it/s]


In [8]:
print("\nDONE")
for k, v in counter.items():
    print(f"{k}: {v} campioni salvati")


DONE
train: 4320 campioni salvati
val: 120 campioni salvati
test: 240 campioni salvati
